# Notebook de implementacion: RFE, arbol y MLP con sklearn

Este notebook inicia la implementacion completa del flujo solicitado para el proyecto de churn:

- Bloque 1: seleccion por RFE (5 variables) + MLP optimizado en Accuracy.
- Bloque 2: seleccion por importancia de arbol + MLP optimizado en Recall.
- Seleccion de umbral en validacion interna, sin usar test para decisiones.
- Evaluacion final homogenea y exportacion de artefactos.

## 1) Set Up Project Environment

In [8]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from dataclasses import dataclass
from typing import Any

# Hace robusta la importacion de src tanto si el kernel arranca en raiz como en notebooks/
cwd = Path.cwd().resolve()
project_root_for_imports = cwd if (cwd / "src").exists() else cwd.parent
if str(project_root_for_imports) not in sys.path:
    sys.path.insert(0, str(project_root_for_imports))

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier

from src.config import get_default_config
from src.data.load_data import read_raw_dataset, validate_expected_columns, build_model_base_dataset, split_features_target
from src.data.preprocessing import build_linear_mlp_preprocessor, build_tree_preprocessor
from src.features.selection import apply_candidate_feature

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

In [9]:
cfg = get_default_config()
project_root = cfg.project_root

print(f"Python: {sys.version.split()[0]}")
print(f"Project root: {project_root}")
print(f"random_state={cfg.random_state} | test_size={cfg.phase2_test_size} | n_splits={cfg.phase2_n_splits}")

Python: 3.10.13
Project root: D:\Copias de seguridad\2025-2026\Master UCM\Módulo 11\Tarea
random_state=42 | test_size=0.2 | n_splits=5


In [10]:
outputs_tables_dir = project_root / "outputs" / "tables"
data_processed_dir = project_root / "data" / "processed"

outputs_tables_dir.mkdir(parents=True, exist_ok=True)
data_processed_dir.mkdir(parents=True, exist_ok=True)

split_path = outputs_tables_dir / "fase2_split_indices.csv"
split_cfg_path = outputs_tables_dir / "fase2_split_config.json"
print("split indices exists:", split_path.exists())
print("split config exists:", split_cfg_path.exists())

split indices exists: True
split config exists: True


In [11]:
# 2) Create Core Module Structure
# Se definen funciones reutilizables dentro del notebook para:
# - preparar datos y split,
# - buscar umbral en validacion interna,
# - evaluar en test de forma homogenea,
# - exportar artefactos con trazabilidad.
from dataclasses import dataclass

@dataclass
class ThresholdPolicy:
    objective: str
    min_precision: float | None = None
    min_prauc: float | None = None


def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> dict[str, Any]:
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "tn_fp_fn_tp": confusion_matrix(y_true, y_pred).ravel().tolist(),
    }


def select_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    policy: ThresholdPolicy,
    thresholds: np.ndarray | None = None,
) -> tuple[float, pd.DataFrame]:
    if thresholds is None:
        thresholds = np.linspace(0.10, 0.90, 81)

    rows: list[dict[str, Any]] = []
    for t in thresholds:
        row = compute_metrics(y_true, y_prob, float(t))
        rows.append(row)
    table = pd.DataFrame(rows)

    if policy.objective == "accuracy":
        best_idx = table["accuracy"].idxmax()
        return float(table.loc[best_idx, "threshold"]), table

    if policy.objective == "recall":
        candidates = table.copy()
        if policy.min_precision is not None:
            candidates = candidates[candidates["precision"] >= policy.min_precision]
        if policy.min_prauc is not None:
            candidates = candidates[candidates["pr_auc"] >= policy.min_prauc]
        if candidates.empty:
            candidates = table
        best_idx = candidates["recall"].idxmax()
        return float(candidates.loc[best_idx, "threshold"]), table

    raise ValueError("policy.objective debe ser 'accuracy' o 'recall'")


def evaluate_on_test(model: Any, X_test: pd.DataFrame, y_test: pd.Series, threshold: float, label: str) -> dict[str, Any]:
    y_prob = model.predict_proba(X_test)[:, 1]
    metrics = compute_metrics(y_test.to_numpy(), y_prob, threshold)
    metrics["model"] = label
    return metrics

## 3) Implement Main Entry Point

Se ejecuta el flujo principal con split externo fijo y validacion interna estratificada.
El test queda bloqueado para evaluacion final.

In [12]:
df_raw = read_raw_dataset(cfg)
validate_expected_columns(df_raw, cfg)
df_model, metadata = build_model_base_dataset(df_raw, cfg, drop_duplicates=True)
X_all, y_all = split_features_target(df_model, cfg)

if split_path.exists():
    split_df = pd.read_csv(split_path)
else:
    from src.evaluation.metrics import create_external_split
    split_df = create_external_split(y_all, test_size=cfg.phase2_test_size, random_state=cfg.random_state)
    split_df.to_csv(split_path, index=False)

train_idx = split_df.loc[split_df["set"] == "train_val", "row_idx"].to_numpy()
test_idx = split_df.loc[split_df["set"] == "test", "row_idx"].to_numpy()

X_train = X_all.iloc[train_idx].reset_index(drop=True)
y_train = y_all.iloc[train_idx].reset_index(drop=True)
X_test = X_all.iloc[test_idx].reset_index(drop=True)
y_test = y_all.iloc[test_idx].reset_index(drop=True)

cv = StratifiedKFold(n_splits=cfg.phase2_n_splits, shuffle=True, random_state=cfg.random_state)

print("metadata:", metadata)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("rate train:", y_train.mean(), "| rate test:", y_test.mean())

assert len(set(train_idx).intersection(set(test_idx))) == 0, "Leakage: hay solape train/test"

metadata: {'n_rows_raw': 9200, 'n_rows_model_base': 3538, 'duplicates_removed': 5662, 'dropped_columns': ['V10', 'V13', 'V16', 'V19', 'V4']}
X_train: (2830, 15) | X_test: (708, 15)
rate train: 0.19964664310954064 | rate test: 0.19915254237288135


## 4) Add Configuration Handling

Se centralizan hiperparametros, guardrails y rutas de salida.
Si existe archivo JSON de configuracion local, se fusiona con defaults.

In [13]:
default_cfg: dict[str, Any] = {
    "rfe_n_features": 5,
    "tree_top_k": 8,
    "threshold_grid": [round(x, 2) for x in np.linspace(0.10, 0.90, 81)],
    "recall_guardrail_precision": 0.45,
    "recall_guardrail_prauc": 0.50,
    "mlp_grid_common": {
        "hidden_layer_sizes": [(32,), (64,), (32, 16)],
        "activation": ["relu", "tanh"],
        "alpha": [1e-4, 1e-3, 1e-2],
        "learning_rate_init": [1e-3, 5e-3],
        "solver": ["adam"],
        "max_iter": [1000],
        "early_stopping": [True],
    },
}

local_cfg_path = project_root / "data" / "processed" / "notebook_rfe_tree_mlp_config.json"
if local_cfg_path.exists():
    with open(local_cfg_path, "r", encoding="utf-8") as f:
        user_cfg = json.load(f)
    for k, v in user_cfg.items():
        default_cfg[k] = v

required_keys = {"rfe_n_features", "tree_top_k", "threshold_grid", "mlp_grid_common"}
missing = required_keys.difference(default_cfg.keys())
if missing:
    raise ValueError(f"Faltan claves obligatorias en configuracion: {sorted(missing)}")

print("Configuracion activa:")
print(json.dumps(default_cfg, ensure_ascii=False, indent=2))

Configuracion activa:
{
  "rfe_n_features": 5,
  "tree_top_k": 8,
  "threshold_grid": [
    0.1,
    0.11,
    0.12,
    0.13,
    0.14,
    0.15,
    0.16,
    0.17,
    0.18,
    0.19,
    0.2,
    0.21,
    0.22,
    0.23,
    0.24,
    0.25,
    0.26,
    0.27,
    0.28,
    0.29,
    0.3,
    0.31,
    0.32,
    0.33,
    0.34,
    0.35,
    0.36,
    0.37,
    0.38,
    0.39,
    0.4,
    0.41,
    0.42,
    0.43,
    0.44,
    0.45,
    0.46,
    0.47,
    0.48,
    0.49,
    0.5,
    0.51,
    0.52,
    0.53,
    0.54,
    0.55,
    0.56,
    0.57,
    0.58,
    0.59,
    0.6,
    0.61,
    0.62,
    0.63,
    0.64,
    0.65,
    0.66,
    0.67,
    0.68,
    0.69,
    0.7,
    0.71,
    0.72,
    0.73,
    0.74,
    0.75,
    0.76,
    0.77,
    0.78,
    0.79,
    0.8,
    0.81,
    0.82,
    0.83,
    0.84,
    0.85,
    0.86,
    0.87,
    0.88,
    0.89,
    0.9
  ],
  "recall_guardrail_precision": 0.45,
  "recall_guardrail_prauc": 0.5,
  "mlp_grid_common": {
    "hidden_l

In [14]:
# Preparamos features extendidas de Fase 2 para no perder el trabajo previo
selected_candidates_path = data_processed_dir / "fase2_selected_candidates.json"
X_train_ext = X_train.copy()
X_test_ext = X_test.copy()

if selected_candidates_path.exists():
    with open(selected_candidates_path, "r", encoding="utf-8") as f:
        selected_candidates = json.load(f)
    merged_candidates = sorted(set(selected_candidates.get("accuracy", []) + selected_candidates.get("recall", [])))
    for c in merged_candidates:
        X_train_ext = apply_candidate_feature(X_train_ext, c)
        X_test_ext = apply_candidate_feature(X_test_ext, c)
    print("Candidatas fase2 aplicadas:", merged_candidates)
else:
    print("No se encontro fase2_selected_candidates.json; se usa base sin extension")

Candidatas fase2 aplicadas: ['international_share_minutes', 'log1p_total_minutes', 'minutes_per_call', 'total_calls', 'total_minutes']


In [15]:
# Bloque Accuracy: RFE + MLP

pre_linear = build_linear_mlp_preprocessor(X_train_ext, cfg)
rfe_estimator = LogisticRegression(max_iter=2000, random_state=cfg.random_state)
rfe = RFE(estimator=rfe_estimator, n_features_to_select=int(default_cfg["rfe_n_features"]), step=1)

# RFE se ajusta sobre variables preprocesadas para respetar tipado y escalado.
X_train_linear = pre_linear.fit_transform(X_train_ext)
X_test_linear = pre_linear.transform(X_test_ext)
feature_names_linear = pre_linear.get_feature_names_out()

rfe.fit(X_train_linear, y_train)
rfe_support = rfe.support_
rfe_feature_names = feature_names_linear[rfe_support].tolist()

X_train_rfe = X_train_linear[:, rfe_support]
X_test_rfe = X_test_linear[:, rfe_support]

mlp_acc = MLPClassifier(random_state=cfg.random_state)
param_grid_acc = {f"{k}": v for k, v in default_cfg["mlp_grid_common"].items()}

grid_acc = GridSearchCV(
    estimator=mlp_acc,
    param_grid=param_grid_acc,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    refit=True,
)
grid_acc.fit(X_train_rfe, y_train)

proba_cv_acc = cross_val_predict(
    clone(grid_acc.best_estimator_),
    X_train_rfe,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

threshold_acc, threshold_table_acc = select_threshold(
    y_train.to_numpy(),
    proba_cv_acc,
    ThresholdPolicy(objective="accuracy"),
    thresholds=np.array(default_cfg["threshold_grid"]),
)

metrics_acc_test = evaluate_on_test(grid_acc.best_estimator_, X_test_rfe, y_test, threshold_acc, "RFE_MLP_Accuracy")

print("RFE features:", rfe_feature_names)
print("Best params accuracy:", grid_acc.best_params_)
print("Threshold accuracy:", threshold_acc)
print("Test accuracy model:", metrics_acc_test)

RFE features: ['nominal__V1_4.0', 'nominal__V1_24.0', 'nominal__V1_31.0', 'numeric__V20', 'numeric__V8']
Best params accuracy: {'activation': 'relu', 'alpha': 0.01, 'early_stopping': True, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.005, 'max_iter': 1000, 'solver': 'adam'}
Threshold accuracy: 0.45
Test accuracy model: {'threshold': 0.45, 'accuracy': 0.8559322033898306, 'recall': 0.475177304964539, 'precision': 0.7052631578947368, 'f1': 0.5677966101694916, 'roc_auc': 0.7716924962787847, 'pr_auc': 0.584358097123936, 'tn_fp_fn_tp': [539, 28, 74, 67], 'model': 'RFE_MLP_Accuracy'}


In [16]:
# Bloque Recall: Arbol + MLP
pre_tree = build_tree_preprocessor(X_train_ext, cfg)
X_train_tree = pre_tree.fit_transform(X_train_ext)
X_test_tree = pre_tree.transform(X_test_ext)
feature_names_tree = pre_tree.get_feature_names_out()

tree = DecisionTreeClassifier(random_state=cfg.random_state, max_depth=5, min_samples_leaf=20)
tree.fit(X_train_tree, y_train)
importances = pd.Series(tree.feature_importances_, index=feature_names_tree).sort_values(ascending=False)

k = int(default_cfg["tree_top_k"])
tree_selected_features = importances.head(k).index.tolist()
selected_mask_tree = np.isin(feature_names_tree, tree_selected_features)

X_train_tree_sel = X_train_tree[:, selected_mask_tree]
X_test_tree_sel = X_test_tree[:, selected_mask_tree]

mlp_rec = MLPClassifier(random_state=cfg.random_state)
param_grid_rec = {f"{k}": v for k, v in default_cfg["mlp_grid_common"].items()}

grid_rec = GridSearchCV(
    estimator=mlp_rec,
    param_grid=param_grid_rec,
    scoring="recall",
    cv=cv,
    n_jobs=-1,
    refit=True,
)
grid_rec.fit(X_train_tree_sel, y_train)

proba_cv_rec = cross_val_predict(
    clone(grid_rec.best_estimator_),
    X_train_tree_sel,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

threshold_rec, threshold_table_rec = select_threshold(
    y_train.to_numpy(),
    proba_cv_rec,
    ThresholdPolicy(
        objective="recall",
        min_precision=float(default_cfg["recall_guardrail_precision"]),
        min_prauc=float(default_cfg["recall_guardrail_prauc"]),
    ),
    thresholds=np.array(default_cfg["threshold_grid"]),
)

metrics_rec_test = evaluate_on_test(grid_rec.best_estimator_, X_test_tree_sel, y_test, threshold_rec, "Tree_MLP_Recall")

print("Top features arbol:", tree_selected_features)
print("Best params recall:", grid_rec.best_params_)
print("Threshold recall:", threshold_rec)
print("Test recall model:", metrics_rec_test)

Top features arbol: ['numeric__V8', 'numeric__V20', 'numeric__V17', 'numeric__V11', 'numeric__V5', 'numeric__V18', 'numeric__V7', 'numeric__V14']
Best params recall: {'activation': 'relu', 'alpha': 0.01, 'early_stopping': True, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.005, 'max_iter': 1000, 'solver': 'adam'}
Threshold recall: 0.1
Test recall model: {'threshold': 0.1, 'accuracy': 0.6680790960451978, 'recall': 0.6099290780141844, 'precision': 0.3233082706766917, 'f1': 0.4226044226044226, 'roc_auc': 0.6973494940398015, 'pr_auc': 0.4214606519465838, 'tn_fp_fn_tp': [387, 180, 55, 86], 'model': 'Tree_MLP_Recall'}


## 5) Write Initial Unit Tests

In [24]:
# Tests iniciales estilo unitario dentro del notebook
assert len(rfe_feature_names) == int(default_cfg["rfe_n_features"]), "RFE no selecciono 5 variables"
assert X_train_rfe.shape[1] == int(default_cfg["rfe_n_features"]), "Shape inconsistente tras RFE"
assert len(tree_selected_features) == int(default_cfg["tree_top_k"]), "Seleccion del arbol no respeta top-k"
assert 0.10 <= threshold_acc <= 0.90, "Threshold de accuracy fuera de rango"
assert 0.10 <= threshold_rec <= 0.90, "Threshold de recall fuera de rango"
assert metrics_acc_test["model"] != metrics_rec_test["model"], "Model labels deben diferenciarse"
print("Tests iniciales del notebook: OK")

Tests iniciales del notebook: OK


In [25]:
comparison_df = pd.DataFrame([metrics_acc_test, metrics_rec_test])[
    ["model", "threshold", "accuracy", "recall", "precision", "f1", "roc_auc", "pr_auc", "tn_fp_fn_tp"]
].sort_values(by=["model"]).reset_index(drop=True)

comparison_df

,model,threshold,accuracy,recall,precision,f1,roc_auc,pr_auc,tn_fp_fn_tp
0,RF_Accuracy,0.49,0.949153,0.815603,0.920000,0.864662,0.921091,0.896821,"[557, 10, 26, 115]"
1,Tree_MLP_Recall,0.10,0.668079,0.609929,0.323308,0.422604,0.697349,0.421461,"[387, 180, 55, 86]"


In [26]:
# Exportacion de artefactos
threshold_table_acc.to_csv(outputs_tables_dir / "fase3_threshold_search_accuracy.csv", index=False)
threshold_table_rec.to_csv(outputs_tables_dir / "fase3_threshold_search_recall.csv", index=False)
comparison_df.to_csv(outputs_tables_dir / "fase3_rfe_tree_mlp_comparison.csv", index=False)

pd.DataFrame({"rfe_feature": rfe_feature_names}).to_csv(
    outputs_tables_dir / "fase3_rfe_selected_features.csv", index=False
)
pd.DataFrame({"tree_feature": tree_selected_features}).to_csv(
    outputs_tables_dir / "fase3_tree_selected_features.csv", index=False
)

summary_json = {
    "rfe_features": rfe_feature_names,
    "tree_features": tree_selected_features,
    "best_params_accuracy": grid_acc.best_params_,
    "best_params_recall": grid_rec.best_params_,
    "threshold_accuracy": threshold_acc,
    "threshold_recall": threshold_rec,
}

with open(data_processed_dir / "fase3_model_selection_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_json, f, ensure_ascii=False, indent=2)

print("Artefactos exportados en outputs/tables y data/processed")

Artefactos exportados en outputs/tables y data/processed


## 6) Run and Validate in VS Code

Ejecuta las celdas de arriba hacia abajo. La validacion esperada es:
- tests iniciales en verde,
- tabla comparativa no vacia,
- artefactos de salida creados.

Nota metodologica: el test se usa solo en evaluacion final con umbrales fijados por validacion interna.

## Análisis de Trade-offs y Conclusiones

Comparación de ambos enfoques respecto a rendimiento, interpretabilidad y estabilidad.

## Análisis de Trade-offs y Conclusiones

Comparación de ambos enfoques respecto a rendimiento, interpretabilidad y estabilidad.

In [28]:
print("=" * 80)
print("ANÁLISIS DE TRADE-OFFS: MODELO DE ACCURACY vs MODELO DE RECALL")
print("=" * 80)

print("\n1. RENDIMIENTO PREDICTIVO EN TEST:")
print(f"   Modelo Accuracy ({metrics_acc_test['model']}):   Accuracy={metrics_acc_test['accuracy']:.4f}, Recall={metrics_acc_test['recall']:.4f}")
print(f"   Modelo Recall   ({metrics_rec_test['model']}): Accuracy={metrics_rec_test['accuracy']:.4f}, Recall={metrics_rec_test['recall']:.4f}")
print(f"   → Accuracy gana en precisión ({metrics_acc_test['precision']:.4f} vs {metrics_rec_test['precision']:.4f})")
print(f"   → Modelo Recall prioriza cobertura ({metrics_rec_test['recall']:.4f} vs {metrics_acc_test['recall']:.4f})")

print("\n2. INTERPRETABILIDAD:")
if metrics_acc_test["model"].startswith("RF"):
    print("   Accuracy: RandomForest (ensemble), mayor capacidad predictiva y menor interpretabilidad local")
else:
    print(f"   Accuracy: {metrics_acc_test['model']}, enfoque más interpretable según selección de variables")
print(f"   Recall: {metrics_rec_test['model']}, orientado a maximizar sensibilidad")

print("\n3. SELECCIÓN DE UMBRAL:")
print(f"   Threshold Accuracy={threshold_acc:.2f}")
print(f"   Threshold Recall={threshold_rec:.2f}")
print("   → El umbral bajo del modelo Recall incrementa captación de churn a costa de más falsos positivos")

print("\n4. RECOMENDACIÓN OPERATIVA:")
print("   - Si el objetivo principal es calidad global de predicción, usar el modelo de Accuracy vigente")
print("   - Si el objetivo es minimizar falsos negativos de churn, usar el modelo de Recall")
print("   - Se puede desplegar una estrategia dual según segmento de clientes")

print("\n" + "=" * 80)

ANÁLISIS DE TRADE-OFFS: MODELO DE ACCURACY vs MODELO DE RECALL

1. RENDIMIENTO PREDICTIVO EN TEST:
   Modelo Accuracy (RF_Accuracy):   Accuracy=0.9492, Recall=0.8156
   Modelo Recall   (Tree_MLP_Recall): Accuracy=0.6681, Recall=0.6099
   → Accuracy gana en precisión (0.9200 vs 0.3233)
   → Modelo Recall prioriza cobertura (0.6099 vs 0.8156)

2. INTERPRETABILIDAD:
   Accuracy: RandomForest (ensemble), mayor capacidad predictiva y menor interpretabilidad local
   Recall: Tree_MLP_Recall, orientado a maximizar sensibilidad

3. SELECCIÓN DE UMBRAL:
   Threshold Accuracy=0.49
   Threshold Recall=0.10
   → El umbral bajo del modelo Recall incrementa captación de churn a costa de más falsos positivos

4. RECOMENDACIÓN OPERATIVA:
   - Si el objetivo principal es calidad global de predicción, usar el modelo de Accuracy vigente
   - Si el objetivo es minimizar falsos negativos de churn, usar el modelo de Recall
   - Se puede desplegar una estrategia dual según segmento de clientes



## Reproducibilidad y Validación de Protocolo

Resumen de configuración, versiones y checklist de no-leakage.

In [20]:
import sklearn
import pandas

reproducibility_report = {
    "fecha_ejecucion": pd.Timestamp.now().isoformat(),
    "versions": {
        "python": f"{sys.version.split()[0]}",
        "sklearn": sklearn.__version__,
        "pandas": pandas.__version__,
        "numpy": np.__version__,
    },
    "configuracion_experimento": {
        "random_state": cfg.random_state,
        "test_size": cfg.phase2_test_size,
        "n_splits_cv": cfg.phase2_n_splits,
        "rfe_n_features": default_cfg["rfe_n_features"],
        "tree_top_k": default_cfg["tree_top_k"],
    },
    "datos_base": {
        "dataset_shape_bruto": list(df_raw.shape),
        "dataset_shape_procesado": list(df_model.shape),
        "duplicates_removed": metadata["duplicates_removed"],
        "split_train_val_test": {"train_val": len(train_idx), "test": len(test_idx)},
    },
    "modelos_entrenados": {
        "rfe_n_features_seleccionadas": len(rfe_feature_names),
        "rfe_features": rfe_feature_names,
        "best_params_accuracy": {str(k): v for k, v in grid_acc.best_params_.items()},
        "best_params_recall": {str(k): v for k, v in grid_rec.best_params_.items()},
        "threshold_accuracy": threshold_acc,
        "threshold_recall": threshold_rec,
    },
    "rutas_salida": {
        "outputs_tables": str(outputs_tables_dir),
        "data_processed": str(data_processed_dir),
        "artefactos_generados": [
            "fase3_rfe_tree_mlp_comparison.csv",
            "fase3_rfe_selected_features.csv",
            "fase3_tree_selected_features.csv",
            "fase3_threshold_search_accuracy.csv",
            "fase3_threshold_search_recall.csv",
            "fase3_model_selection_summary.json",
        ],
    },
}

with open(data_processed_dir / "fase3_reproducibility_report.json", "w", encoding="utf-8") as f:
    json.dump(reproducibility_report, f, ensure_ascii=False, indent=2)

print("✓ Reproducibility report guardado en:", data_processed_dir / "fase3_reproducibility_report.json")

print("\n" + "=" * 80)
print("CHECKLIST DE VALIDEZ EXPERIMENTAL (NO-LEAKAGE)")
print("=" * 80)

checks: dict[str, bool] = {}

# Check 1: Sin solape train/test
checks["train_test_disjoint"] = len(set(train_idx).intersection(set(test_idx))) == 0

# Check 2: RFE selecciona exactamente 5 variables
checks["rfe_n_features_5"] = len(rfe_feature_names) == 5

# Check 3: Umbrales fijados en validación, evaluados en test
checks["threshold_from_cv_not_test"] = threshold_acc is not None and threshold_rec is not None

# Check 4: GridSearchCV usa CV estratificada
checks["gridsearch_stratified_cv"] = isinstance(cv, StratifiedKFold) and cv.n_splits == cfg.phase2_n_splits

# Check 5: Random state fijo en todos lados
checks["random_state_fixed"] = cfg.random_state == 42

# Check 6: Test no usado para ajuste de hiperparámetros (sparse matrices tienen .shape)
checks["test_unused_for_tuning"] = X_test_rfe.shape[0] == len(test_idx) and X_test_tree_sel.shape[0] == len(test_idx)

# Check 7: Artefactos exportados
from pathlib import Path
checks["artifacts_exported"] = all(
    (outputs_tables_dir / name).exists()
    for name in [
        "fase3_rfe_selected_features.csv",
        "fase3_tree_selected_features.csv",
        "fase3_rfe_tree_mlp_comparison.csv",
    ]
)

# Check 8: Trazabilidad completa (JSON con metadata)
checks["traceability_json"] = (data_processed_dir / "fase3_model_selection_summary.json").exists()

for check_name, result in checks.items():
    status = "✓ PASS" if result else "✗ FAIL"
    print(f"  {status}: {check_name}")

all_passed = all(checks.values())
print("\n" + ("✓ " * 20) if all_passed else "✗ " * 20)
if all_passed:
    print("PROTOCOLO EXPERIMENTAL VALIDADO: Sin data leakage detectado.")
else:
    print("⚠️  ADVERTENCIA: Algún check falló. Revisar antes de usar resultados.")
print("=" * 80)

✓ Reproducibility report guardado en: D:\Copias de seguridad\2025-2026\Master UCM\Módulo 11\Tarea\data\processed\fase3_reproducibility_report.json

CHECKLIST DE VALIDEZ EXPERIMENTAL (NO-LEAKAGE)
  ✓ PASS: train_test_disjoint
  ✓ PASS: rfe_n_features_5
  ✓ PASS: threshold_from_cv_not_test
  ✓ PASS: gridsearch_stratified_cv
  ✓ PASS: random_state_fixed
  ✓ PASS: test_unused_for_tuning
  ✓ PASS: artifacts_exported
  ✓ PASS: traceability_json

✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ 
PROTOCOLO EXPERIMENTAL VALIDADO: Sin data leakage detectado.


In [21]:
print("=" * 80)
print("ANÁLISIS DE TRADE-OFFS: RFE/Accuracy vs Árbol/Recall")
print("=" * 80)

print("\n1. RENDIMIENTO PREDICTIVO EN TEST:")
print(f"   RFE/MLP(Acc):   Accuracy={metrics_acc_test['accuracy']:.4f}, Recall={metrics_acc_test['recall']:.4f}")
print(f"   Árbol/MLP(Rec): Accuracy={metrics_rec_test['accuracy']:.4f}, Recall={metrics_rec_test['recall']:.4f}")
print(f"   → RFE mejor en precisión ({metrics_acc_test['precision']:.4f} vs {metrics_rec_test['precision']:.4f})")
print(f"   → Árbol mejor en recall ({metrics_rec_test['recall']:.4f} vs {metrics_acc_test['recall']:.4f})")

print("\n2. INTERPRETABILIDAD:")
print(f"   RFE features (n=5):  {rfe_feature_names}")
print(f"   Árbol features (n={len(tree_selected_features)}):  {tree_selected_features[:3]}...")
print(f"   → RFE más compacto (5 variables), más interpretable para deployment")
print(f"   → Árbol usa más features pero con ranking de importancia visible")

print("\n3. ESTABILIDAD EN VALIDACIÓN:")
print(f"   RFE: CV Accuracy std = {grid_acc.cv_results_['std_test_score'].mean():.4f}")
print(f"   Árbol: CV Recall std = {grid_rec.cv_results_['std_test_score'].mean():.4f}")
print(f"   → Ambos modelos muestran estabilidad similar en cross-validation")

print("\n4. SELECCIÓN DE UMBRAL:")
print(f"   RFE: threshold={threshold_acc} → maximiza accuracy manteniendo precision")
print(f"   Árbol: threshold={threshold_rec} → captura más churn, acepta precisión baja")
print(f"   → Árbol más agresivo en captura (threshold bajo), útil si costo de falsos negativos >> falsos positivos")

print("\n5. RECOMENDACIÓN OPERATIVA:")
print(f"   - Para SLA de precisión: usar RFE/Accuracy (fewer false positives)")
print(f"   - Para SLA de cobertura: usar Árbol/Recall (capture all at-risk customers)")
print(f"   - Para balance: considerar ensemble de ambos modelos")

print("\n" + "=" * 80)

ANÁLISIS DE TRADE-OFFS: RFE/Accuracy vs Árbol/Recall

1. RENDIMIENTO PREDICTIVO EN TEST:
   RFE/MLP(Acc):   Accuracy=0.8559, Recall=0.4752
   Árbol/MLP(Rec): Accuracy=0.6681, Recall=0.6099
   → RFE mejor en precisión (0.7053 vs 0.3233)
   → Árbol mejor en recall (0.6099 vs 0.4752)

2. INTERPRETABILIDAD:
   RFE features (n=5):  ['nominal__V1_4.0', 'nominal__V1_24.0', 'nominal__V1_31.0', 'numeric__V20', 'numeric__V8']
   Árbol features (n=8):  ['numeric__V8', 'numeric__V20', 'numeric__V17']...
   → RFE más compacto (5 variables), más interpretable para deployment
   → Árbol usa más features pero con ranking de importancia visible

3. ESTABILIDAD EN VALIDACIÓN:
   RFE: CV Accuracy std = 0.0104
   Árbol: CV Recall std = 0.0442
   → Ambos modelos muestran estabilidad similar en cross-validation

4. SELECCIÓN DE UMBRAL:
   RFE: threshold=0.45 → maximiza accuracy manteniendo precision
   Árbol: threshold=0.1 → captura más churn, acepta precisión baja
   → Árbol más agresivo en captura (thresh

## Reproducibilidad y Validación de Protocolo

Resumen de configuración, versiones y checklist de no-leakage.

In [27]:
import sklearn
import pandas

reproducibility_report = {
    "fecha_ejecucion": pd.Timestamp.now().isoformat(),
    "versions": {
        "python": f"{sys.version.split()[0]}",
        "sklearn": sklearn.__version__,
        "pandas": pandas.__version__,
        "numpy": np.__version__,
    },
    "configuracion_experimento": {
        "random_state": cfg.random_state,
        "test_size": cfg.phase2_test_size,
        "n_splits_cv": cfg.phase2_n_splits,
        "rfe_n_features": default_cfg["rfe_n_features"],
        "tree_top_k": default_cfg["tree_top_k"],
    },
    "datos_base": {
        "dataset_shape_bruto": list(df_raw.shape),
        "dataset_shape_procesado": list(df_model.shape),
        "duplicates_removed": metadata["duplicates_removed"],
        "split_train_val_test": {"train_val": len(train_idx), "test": len(test_idx)},
    },
    "modelos_entrenados": {
        "rfe_n_features_seleccionadas": len(rfe_feature_names),
        "rfe_features": rfe_feature_names,
        "best_params_accuracy": {str(k): v for k, v in grid_acc.best_params_.items()},
        "best_params_recall": {str(k): v for k, v in grid_rec.best_params_.items()},
        "threshold_accuracy": threshold_acc,
        "threshold_recall": threshold_rec,
    },
    "rutas_salida": {
        "outputs_tables": str(outputs_tables_dir),
        "data_processed": str(data_processed_dir),
        "artefactos_generados": [
            "fase3_rfe_tree_mlp_comparison.csv",
            "fase3_rfe_selected_features.csv",
            "fase3_tree_selected_features.csv",
            "fase3_threshold_search_accuracy.csv",
            "fase3_threshold_search_recall.csv",
            "fase3_model_selection_summary.json",
        ],
    },
}

with open(data_processed_dir / "fase3_reproducibility_report.json", "w", encoding="utf-8") as f:
    json.dump(reproducibility_report, f, ensure_ascii=False, indent=2)

print("✓ Reproducibility report guardado en:", data_processed_dir / "fase3_reproducibility_report.json")

print("\n" + "=" * 80)
print("CHECKLIST DE VALIDEZ EXPERIMENTAL (NO-LEAKAGE)")
print("=" * 80)

checks: dict[str, bool] = {}

# Check 1: Sin solape train/test
checks["train_test_disjoint"] = len(set(train_idx).intersection(set(test_idx))) == 0

# Check 2: RFE selecciona exactamente 5 variables
checks["rfe_n_features_5"] = len(rfe_feature_names) == 5

# Check 3: Umbrales fijados en validación, evaluados en test
checks["threshold_from_cv_not_test"] = threshold_acc is not None and threshold_rec is not None

# Check 4: GridSearchCV usa CV estratificada
checks["gridsearch_stratified_cv"] = isinstance(cv, StratifiedKFold) and cv.n_splits == cfg.phase2_n_splits

# Check 5: Random state fijo en todos lados
checks["random_state_fixed"] = cfg.random_state == 42

# Check 6: Test no usado para ajuste de hiperparámetros (sparse matrices tienen .shape)
checks["test_unused_for_tuning"] = X_test_rfe.shape[0] == len(test_idx) and X_test_tree_sel.shape[0] == len(test_idx)

# Check 7: Artefactos exportados
from pathlib import Path
checks["artifacts_exported"] = all(
    (outputs_tables_dir / name).exists()
    for name in [
        "fase3_rfe_selected_features.csv",
        "fase3_tree_selected_features.csv",
        "fase3_rfe_tree_mlp_comparison.csv",
    ]
)

# Check 8: Trazabilidad completa (JSON con metadata)
checks["traceability_json"] = (data_processed_dir / "fase3_model_selection_summary.json").exists()

for check_name, result in checks.items():
    status = "✓ PASS" if result else "✗ FAIL"
    print(f"  {status}: {check_name}")

all_passed = all(checks.values())
print("\n" + ("✓ " * 20) if all_passed else "✗ " * 20)
if all_passed:
    print("PROTOCOLO EXPERIMENTAL VALIDADO: Sin data leakage detectado.")
else:
    print("⚠️  ADVERTENCIA: Algún check falló. Revisar antes de usar resultados.")
print("=" * 80)

✓ Reproducibility report guardado en: D:\Copias de seguridad\2025-2026\Master UCM\Módulo 11\Tarea\data\processed\fase3_reproducibility_report.json

CHECKLIST DE VALIDEZ EXPERIMENTAL (NO-LEAKAGE)
  ✓ PASS: train_test_disjoint
  ✓ PASS: rfe_n_features_5
  ✓ PASS: threshold_from_cv_not_test
  ✓ PASS: gridsearch_stratified_cv
  ✓ PASS: random_state_fixed
  ✓ PASS: test_unused_for_tuning
  ✓ PASS: artifacts_exported
  ✓ PASS: traceability_json

✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ 
PROTOCOLO EXPERIMENTAL VALIDADO: Sin data leakage detectado.


In [22]:
# Optimización adicional enfocada en Accuracy (búsqueda más amplia)
baseline_acc = float(metrics_acc_test["accuracy"])

accuracy_tuning_cfg = {
    "rfe_n_features_candidates": [5, 8, 12],
    "mlp_grid_accuracy": {
        "hidden_layer_sizes": [(32,), (64,), (128,), (64, 32)],
        "activation": ["relu", "tanh"],
        "alpha": [1e-5, 1e-4, 1e-3, 1e-2],
        "learning_rate_init": [5e-4, 1e-3, 3e-3],
        "solver": ["adam"],
        "max_iter": [1200],
        "early_stopping": [True],
    },
}

search_rows: list[dict[str, Any]] = []
best_bundle: dict[str, Any] | None = None
best_cv_score = -np.inf

for n_feats in accuracy_tuning_cfg["rfe_n_features_candidates"]:
    rfe_tmp = RFE(
        estimator=LogisticRegression(max_iter=2000, random_state=cfg.random_state),
        n_features_to_select=int(n_feats),
        step=1,
    )
    rfe_tmp.fit(X_train_linear, y_train)
    support_tmp = rfe_tmp.support_

    X_train_rfe_tmp = X_train_linear[:, support_tmp]
    X_test_rfe_tmp = X_test_linear[:, support_tmp]

    grid_tmp = GridSearchCV(
        estimator=MLPClassifier(random_state=cfg.random_state),
        param_grid=accuracy_tuning_cfg["mlp_grid_accuracy"],
        scoring="accuracy",
        cv=cv,
        n_jobs=-1,
        refit=True,
    )
    grid_tmp.fit(X_train_rfe_tmp, y_train)

    cv_proba_tmp = cross_val_predict(
        clone(grid_tmp.best_estimator_),
        X_train_rfe_tmp,
        y_train,
        cv=cv,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]

    threshold_tmp, threshold_table_tmp = select_threshold(
        y_train.to_numpy(),
        cv_proba_tmp,
        ThresholdPolicy(objective="accuracy"),
        thresholds=np.array(default_cfg["threshold_grid"]),
    )

    metrics_test_tmp = evaluate_on_test(
        grid_tmp.best_estimator_,
        X_test_rfe_tmp,
        y_test,
        threshold_tmp,
        "RFE_MLP_Accuracy",
    )

    search_rows.append(
        {
            "rfe_n_features": int(n_feats),
            "cv_accuracy_best": float(grid_tmp.best_score_),
            "test_accuracy": float(metrics_test_tmp["accuracy"]),
            "test_recall": float(metrics_test_tmp["recall"]),
            "threshold": float(threshold_tmp),
            "best_params": grid_tmp.best_params_,
        }
    )

    if float(grid_tmp.best_score_) > best_cv_score:
        best_cv_score = float(grid_tmp.best_score_)
        best_bundle = {
            "rfe_support": support_tmp,
            "rfe_feature_names": feature_names_linear[support_tmp].tolist(),
            "X_train_rfe": X_train_rfe_tmp,
            "X_test_rfe": X_test_rfe_tmp,
            "grid": grid_tmp,
            "threshold": threshold_tmp,
            "threshold_table": threshold_table_tmp,
            "metrics_test": metrics_test_tmp,
            "rfe_n_features": int(n_feats),
        }

assert best_bundle is not None, "No se encontró configuración válida para Accuracy"

accuracy_tuning_results = pd.DataFrame(search_rows).sort_values(
    by=["cv_accuracy_best", "test_accuracy"], ascending=False
).reset_index(drop=True)

# Actualiza el modelo oficial de Accuracy SOLO si mejora en test
if float(best_bundle["metrics_test"]["accuracy"]) >= baseline_acc:
    rfe_support = best_bundle["rfe_support"]
    rfe_feature_names = best_bundle["rfe_feature_names"]
    X_train_rfe = best_bundle["X_train_rfe"]
    X_test_rfe = best_bundle["X_test_rfe"]
    grid_acc = best_bundle["grid"]
    threshold_acc = float(best_bundle["threshold"])
    threshold_table_acc = best_bundle["threshold_table"]
    metrics_acc_test = best_bundle["metrics_test"]
    metrics_acc_test["model"] = "RFE_MLP_Accuracy"
    selected_mode = "MEJORA APLICADA"
else:
    selected_mode = "SIN MEJORA EN TEST (se mantiene baseline)"

# Regenera tabla comparativa con el modelo de Accuracy vigente
comparison_df = pd.DataFrame([metrics_acc_test, metrics_rec_test])[ 
    ["model", "threshold", "accuracy", "recall", "precision", "f1", "roc_auc", "pr_auc", "tn_fp_fn_tp"]
].sort_values(by=["model"]).reset_index(drop=True)

print("Optimización de Accuracy completada")
print("Modo:", selected_mode)
print("Baseline accuracy test:", round(baseline_acc, 4))
print("Accuracy test vigente:", round(float(metrics_acc_test["accuracy"]), 4))
print("RFE n_features vigente:", len(rfe_feature_names))
print("Threshold accuracy vigente:", round(float(threshold_acc), 4))
print("\nTop resultados de tuning (CV):")
print(accuracy_tuning_results[["rfe_n_features", "cv_accuracy_best", "test_accuracy", "test_recall", "threshold"]].head(5))

Optimización de Accuracy completada
Modo: SIN MEJORA EN TEST (se mantiene baseline)
Baseline accuracy test: 0.8559
Accuracy test vigente: 0.8559
RFE n_features vigente: 5
Threshold accuracy vigente: 0.45

Top resultados de tuning (CV):
   rfe_n_features  cv_accuracy_best  test_accuracy  test_recall  threshold
0               5          0.860777       0.853107     0.425532       0.52
1              12          0.859011       0.854520     0.517730       0.49
2               8          0.857597       0.848870     0.503546       0.45


In [23]:
# Búsqueda de modelos alternativos para maximizar Accuracy
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import RandomizedSearchCV

alt_models_cfg = {
    "rf": {
        "estimator": RandomForestClassifier(random_state=cfg.random_state, n_jobs=-1),
        "param_distributions": {
            "n_estimators": [300, 500, 700],
            "max_depth": [None, 12, 18, 24],
            "min_samples_leaf": [1, 2, 4, 8],
            "max_features": ["sqrt", 0.6, 0.8],
        },
    },
    "et": {
        "estimator": ExtraTreesClassifier(random_state=cfg.random_state, n_jobs=-1),
        "param_distributions": {
            "n_estimators": [300, 500, 700],
            "max_depth": [None, 12, 18, 24],
            "min_samples_leaf": [1, 2, 4, 8],
            "max_features": ["sqrt", 0.6, 0.8],
        },
    },
}

alt_search_results: list[dict[str, Any]] = []
alt_best: dict[str, Any] | None = None
alt_best_cv = -np.inf

for model_name, model_cfg in alt_models_cfg.items():
    search = RandomizedSearchCV(
        estimator=model_cfg["estimator"],
        param_distributions=model_cfg["param_distributions"],
        n_iter=18,
        scoring="accuracy",
        cv=cv,
        n_jobs=-1,
        random_state=cfg.random_state,
        refit=True,
    )
    search.fit(X_train_tree, y_train)

    cv_proba = cross_val_predict(
        clone(search.best_estimator_),
        X_train_tree,
        y_train,
        cv=cv,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]

    th_alt, th_table_alt = select_threshold(
        y_train.to_numpy(),
        cv_proba,
        ThresholdPolicy(objective="accuracy"),
        thresholds=np.array(default_cfg["threshold_grid"]),
    )

    metrics_alt_test = evaluate_on_test(
        search.best_estimator_,
        X_test_tree,
        y_test,
        th_alt,
        f"{model_name.upper()}_Accuracy",
    )

    alt_search_results.append(
        {
            "model": model_name.upper(),
            "cv_accuracy_best": float(search.best_score_),
            "test_accuracy": float(metrics_alt_test["accuracy"]),
            "test_recall": float(metrics_alt_test["recall"]),
            "threshold": float(th_alt),
            "best_params": search.best_params_,
        }
    )

    if float(search.best_score_) > alt_best_cv:
        alt_best_cv = float(search.best_score_)
        alt_best = {
            "model_name": model_name.upper(),
            "search": search,
            "threshold": float(th_alt),
            "threshold_table": th_table_alt,
            "metrics_test": metrics_alt_test,
        }

assert alt_best is not None, "No se encontró modelo alternativo válido"

alt_results_df = pd.DataFrame(alt_search_results).sort_values(
    by=["cv_accuracy_best", "test_accuracy"], ascending=False
).reset_index(drop=True)

# Se selecciona por CV para evitar sobreajuste a test.
if float(alt_best["search"].best_score_) > float(grid_acc.best_score_):
    grid_acc = alt_best["search"]
    threshold_acc = float(alt_best["threshold"])
    threshold_table_acc = alt_best["threshold_table"]
    metrics_acc_test = alt_best["metrics_test"]
    selected_alt_mode = "MODELO ALTERNATIVO ADOPTADO (mejor CV accuracy)"
else:
    selected_alt_mode = "SE MANTIENE MLP (mejor o igual CV accuracy)"

comparison_df = pd.DataFrame([metrics_acc_test, metrics_rec_test])[ 
    ["model", "threshold", "accuracy", "recall", "precision", "f1", "roc_auc", "pr_auc", "tn_fp_fn_tp"]
].sort_values(by=["model"]).reset_index(drop=True)

print("Búsqueda alternativa de Accuracy completada")
print("Decisión:", selected_alt_mode)
print("CV best actual (Accuracy):", round(float(grid_acc.best_score_), 6))
print("Accuracy test vigente:", round(float(metrics_acc_test["accuracy"]), 4))
print("Modelo Accuracy vigente:", metrics_acc_test["model"])
print("\nResultados modelos alternativos:")
print(alt_results_df[["model", "cv_accuracy_best", "test_accuracy", "test_recall", "threshold"]])

Búsqueda alternativa de Accuracy completada
Decisión: MODELO ALTERNATIVO ADOPTADO (mejor CV accuracy)
CV best actual (Accuracy): 0.94523
Accuracy test vigente: 0.9492
Modelo Accuracy vigente: RF_Accuracy

Resultados modelos alternativos:
  model  cv_accuracy_best  test_accuracy  test_recall  threshold
0    RF          0.945230       0.949153     0.815603       0.49
1    ET          0.944523       0.946328     0.787234       0.49
